In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_score,
    recall_score,
    f1_score
)

import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("Libraries Imported Successfully!")

In [ ]:
# loading data
df = pd.read_csv("Churn_Modelling.csv")
print("Dataset Shape:", df.shape)
print("Churn Rate: {:.2f}%".format(df['Exited'].mean() * 100))
# copy of origina for visualization purpose
df_viz = df.copy() 

In [ ]:
# data cleaning
df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
df = pd.get_dummies(df, columns=['Geography', 'Gender'], drop_first=True)
# features (X) and target variable (y)
X = df.drop("Exited", axis=1)
y = df["Exited"]

In [ ]:
# validation/test
# 80% CV Block / 20% Test
X_cv, X_test, y_cv, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 60% Train / 20% Validation
X_train, X_val, y_train, y_val = train_test_split(
    X_cv, y_cv,
    test_size=0.25,
    random_state=42,
    stratify=y_cv
)

print("Training Size:", X_train.shape[0])
print("Validation Size:", X_val.shape[0])
print("Testing Size:", X_test.shape[0])

In [ ]:
# feature scaling
scaler = StandardScaler()
# columns requiring scaling
columns_to_scale = ['CreditScore', 'Age', 'Balance', 'EstimatedSalary']

X_train[columns_to_scale] = scaler.fit_transform(X_train[columns_to_scale])
X_val[columns_to_scale] = scaler.transform(X_val[columns_to_scale])
X_test[columns_to_scale] = scaler.transform(X_test[columns_to_scale])

In [ ]:
param_grid = {
    'n_estimators': [100, 150],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2']
}

In [ ]:
# model training and hypyerparameter tuning
rf = RandomForestClassifier(
    criterion="entropy",
    random_state=42,
    oob_score=True
)
# optimal hyperparameter
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
# training model on training dataset
grid_search.fit(X_train, y_train)
# extracting best performing model
best_rf = grid_search.best_estimator_

print("Best Parameters:", grid_search.best_params_)
print("Best CV Accuracy:", grid_search.best_score_)

In [ ]:
# model evalutation
train_acc = accuracy_score(y_train, best_rf.predict(X_train))
val_acc   = accuracy_score(y_val, best_rf.predict(X_val))
test_acc  = accuracy_score(y_test, best_rf.predict(X_test))

print("\nACCURACY RESULTS")
print(f"Training Accuracy:   {train_acc*100:.2f}%")
print(f"Validation Accuracy: {val_acc*100:.2f}%")
print(f"Testing Accuracy:    {test_acc*100:.2f}%")

In [ ]:
# precision,recall and f1-score
precision = precision_score(y_test, best_rf.predict(X_test))
recall    = recall_score(y_test, best_rf.predict(X_test))
f1        = f1_score(y_test, best_rf.predict(X_test))

print("\nPrecision:", precision)
print("Recall:", recall)
print("F1-Score:", f1)

In [ ]:
# cross validation performance
cv_scores = cross_val_score(best_rf, X_cv, y_cv, cv=5)

print("\n5-Fold CV Scores:", cv_scores)
print("Mean CV Accuracy: {:.2f}%".format(cv_scores.mean()*100))

In [ ]:
# confusion matrix
from sklearn.metrics import confusion_matrix


cm = confusion_matrix(y_test, best_rf.predict(X_test))

TN, FP, FN, TP = cm.ravel()

plt.figure(figsize=(8,6))

ax = sns.heatmap(cm,
                 annot=False,   
                 fmt='d',
                 cmap='Reds',
                 xticklabels=['Stayed','Churned'],
                 yticklabels=['Stayed','Churned'])


labels = [
    [f"TN\n{TN}", f"FP\n{FP}"],
    [f"FN\n{FN}", f"TP\n{TP}"]
]

for i in range(2):
    for j in range(2):
        ax.text(j + 0.5, i + 0.5,
                labels[i][j],
                ha='center', va='center',
                color='black', fontsize=12)

plt.title("Confusion Matrix (Test Set)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

labels = ["Training", "Validation", "Testing", "Mean CV"]
values = [train_acc*100, val_acc*100, test_acc*100, cv_scores.mean()*100]

plt.figure()
plt.bar(labels, values)

plt.ylabel("Accuracy (%)")
plt.title("Accuracy Comparison Chart")
plt.ylim(0, 100)

for i, v in enumerate(values):
    plt.text(i, v + 1, f"{v:.2f}%", ha='center')

plt.show()

In [ ]:
# feature importance

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": best_rf.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("\nTop 10 Important Features:")
print(feature_importance.head(10))

plt.figure(figsize=(10,6))
plt.barh(feature_importance["Feature"],
         feature_importance["Importance"],
         edgecolor='black')

plt.gca().invert_yaxis()
plt.xlabel("Importance Score")
plt.title("Random Forest Feature Importance")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score


# Predict probabilities for positive class (Churn = 1)
y_prob = best_rf.predict_proba(X_test)[:, 1]

# Calculate ROC curve values
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

# Calculate AUC score
roc_auc = roc_auc_score(y_test, y_prob)

print("ROC-AUC Score:", round(roc_auc, 4))

# Plot ROC Curve
plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, linewidth=2, label=f"AUC = {roc_auc:.4f}")
plt.plot([0,1], [0,1], linestyle='--')  # Random classifier line
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Random Forest (Test Set)")
plt.legend()
plt.show()

In [ ]:

# Geography
plt.figure()
sns.barplot(x="Geography", y="Exited", data=df_viz)
plt.title("Churn Rate by Geography")
plt.show()

# Gender
plt.figure()
sns.barplot(x="Gender", y="Exited", data=df_viz)
plt.title("Churn Rate by Gender")
plt.show()

# Age
plt.figure()
plt.hist(df_viz[df_viz["Exited"]==0]["Age"],
         bins=30, alpha=0.6, label="Stayed")
plt.hist(df_viz[df_viz["Exited"]==1]["Age"],
         bins=30, alpha=0.6, label="Churned")
plt.title("Age Distribution by Churn")
plt.legend()
plt.show()

In [ ]:
# user input prediction

def predict_churn():

    print("\nCustomer Churn Prediction")

    try:
        credit_score = int(input("Credit Score: "))
        geography = input("Geography (France/Germany/Spain): ").capitalize()
        gender = input("Gender (Male/Female): ").capitalize()
        age = int(input("Age: "))
        tenure = int(input("Tenure: "))
        balance = float(input("Balance: "))
        num_products = int(input("Num Products: "))
        has_card = int(input("Has Credit Card (1/0): "))
        is_active = int(input("Is Active (1/0): "))
        salary = float(input("Estimated Salary: "))
    except:
        print("Invalid Input")
        return

    geo_germany = 1 if geography=="Germany" else 0
    geo_spain = 1 if geography=="Spain" else 0
    gender_male = 1 if gender=="Male" else 0

    input_df = pd.DataFrame({
        'CreditScore':[credit_score],
        'Age':[age],
        'Tenure':[tenure],
        'Balance':[balance],
        'NumOfProducts':[num_products],
        'HasCrCard':[has_card],
        'IsActiveMember':[is_active],
        'EstimatedSalary':[salary],
        'Geography_Germany':[geo_germany],
        'Geography_Spain':[geo_spain],
        'Gender_Male':[gender_male]
    })

    input_df[columns_to_scale] = scaler.transform(input_df[columns_to_scale])

    prediction = best_rf.predict(input_df)[0]
    probability = best_rf.predict_proba(input_df)[0][1]*100

    print("\nPrediction:",
          "WILL CHURN" if prediction==1 else "WILL STAY")
    print("Churn Probability:", round(probability,2), "%")


while True:
    predict_churn()
    again = input("\nPredict another? (yes/no): ").lower()
    if again not in ["yes","y"]:
        print("Thank You!")
        break